# HFT Backtest Analysis -- AS-2008 vs AS-2018 vs Naive

Этот ноутбук читает CSV-выгрузки бэктеста (`reports/*_summary.csv`,
`reports/*_timeseries.csv`) и `sample_data/trades_sample.csv`,
и делает три вещи:

1. **Equity curve / inventory path / spread** -- визуальное сравнение стратегий.
2. **Калибровка интенсивности fill** $\lambda(\delta) = A\,e^{-k\delta}$
   на тиковых данных -- значения `A`, `k` для подстановки в `configs/as*.ini`.
3. **Аннуализированные риск-метрики** на минутных бинах (Sharpe, Sortino, max DD).

Запуск (из корня `hft_backtest_integrated/`):

```bash
mkdir -p reports
./build/hft_backtest_integrated configs/as2008.ini
./build/hft_backtest_integrated configs/as2018.ini
./build/hft_backtest_integrated configs/naive.ini
jupyter notebook notebooks/analysis.ipynb
```

<!-- intentionally empty: dedup of the title cell -->

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
REPORTS = ROOT / 'reports'
DATA    = ROOT / 'sample_data'
print('Project root:', ROOT)
print('Reports:    ', sorted(REPORTS.glob('*.csv')))

## 1. Загрузка summary

In [ ]:
def load_summary(path):
    s = pd.read_csv(path)
    return dict(zip(s['metric'], s['value']))

summaries = {}
for label, fname in [
    ('naive',  'naive_summary.csv'),
    ('as2008', 'as2008_summary.csv'),
    ('as2018', 'as2018_summary.csv'),
]:
    p = REPORTS / fname
    if p.exists():
        summaries[label] = load_summary(p)

summary_df = pd.DataFrame(summaries).T
summary_df

## 2. Timeseries -- equity curve, inventory, spread

In [ ]:
def load_timeseries(path):
    df = pd.read_csv(path)
    df['ts'] = pd.to_datetime(df['timestamp_us'], unit='us', utc=True)
    return df

ts = {}
for label, fname in [
    ('naive',  'naive_timeseries.csv'),
    ('as2008', 'as2008_timeseries.csv'),
    ('as2018', 'as2018_timeseries.csv'),
]:
    p = REPORTS / fname
    if p.exists():
        ts[label] = load_timeseries(p)

for k, v in ts.items():
    print(k, v.shape, v['ts'].min(), '->', v['ts'].max())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for label, df in ts.items():
    axes[0].plot(df['ts'], df['total_pnl'], label=label, lw=1)
    axes[1].plot(df['ts'], df['inventory'], label=label, lw=1)
    if 'half_spread' in df:
        axes[2].plot(df['ts'], 2 * df['half_spread'], label=f'{label} quoted', lw=1)
        market_spread = (df['microprice'] - df['mid']).abs() * 2  # грубая прокси
        axes[2].plot(df['ts'], market_spread, label=f'{label} micro-mid', lw=0.5, alpha=0.5)
axes[0].set_title('Equity curve (total PnL, USD)')
axes[1].set_title('Inventory (units)')
axes[2].set_title('Quoted vs proxy market spread')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(loc='best', fontsize=8)
plt.tight_layout(); plt.show()

## 3. Калибровка $\lambda(\delta) = A\,e^{-k\delta}$ по trades.csv

Идея (классика Avellaneda-Stoikov 2008, Section 4): для каждого
пришедшего трейда измеряем расстояние $\delta$ от mid в момент трейда
до цены трейда; число трейдов с $\delta \ge x$ за единицу времени даёт
оценку $\lambda(x)$. Подгоняем линейной регрессией $\log\lambda$ на $\delta$.

In [ ]:
trades = pd.read_csv(DATA / 'trades_sample.csv')
trades.columns = ['idx', 'ts_us', 'side', 'price', 'amount']
trades['ts'] = pd.to_datetime(trades['ts_us'], unit='us', utc=True)
trades.head(3)

In [ ]:
lob = pd.read_csv(DATA / 'lob_sample.csv')
lob = lob.rename(columns={'local_timestamp': 'ts_us'})
lob['mid'] = 0.5 * (lob['asks[0].price'] + lob['bids[0].price'])
lob_small = lob[['ts_us', 'mid']].sort_values('ts_us')
trades_sorted = trades.sort_values('ts_us')
merged = pd.merge_asof(trades_sorted, lob_small, on='ts_us', direction='backward')
merged['delta'] = (merged['price'] - merged['mid']).where(
    merged['side'].str.lower() == 'sell', merged['mid'] - merged['price'])
merged = merged[merged['delta'] > 0]
print('valid trades for calibration:', len(merged))
merged[['ts_us', 'side', 'price', 'mid', 'delta']].head()

In [ ]:
if len(merged) >= 50:
    grid = np.linspace(merged['delta'].quantile(0.05),
                       merged['delta'].quantile(0.95), 30)
    horizon_s = (merged['ts_us'].max() - merged['ts_us'].min()) / 1e6
    lam = np.array([(merged['delta'] >= x).sum() / horizon_s for x in grid])
    mask = lam > 0
    coef = np.polyfit(grid[mask], np.log(lam[mask]), 1)
    k_hat = -coef[0]
    A_hat = np.exp(coef[1])
    print(f'Calibrated  A = {A_hat:.4g}   k = {k_hat:.4g}')
    print('  -> подставьте в configs/as2008.ini, ключ k = ...')
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.semilogy(grid, lam, 'o', label='empirical $\\lambda(\\delta)$')
    ax.semilogy(grid, A_hat * np.exp(-k_hat * grid), '-',
                label=f'fit: {A_hat:.2f}·exp(-{k_hat:.2f}·\\delta)')
    ax.set_xlabel('$\\delta$ (price units from mid)')
    ax.set_ylabel('$\\lambda(\\delta)$ (trades / second)')
    ax.grid(alpha=0.3, which='both'); ax.legend(); plt.tight_layout(); plt.show()
else:
    print('Недостаточно трейдов для устойчивой регрессии. '
          'Запустите на большем фрагменте.')

## 4. Аннуализированные риск-метрики (минутные бины)

In [ ]:
def annualized_metrics(df, periods_per_year=365 * 24 * 60):
    pnl = df['total_pnl'].diff().dropna()
    if len(pnl) < 5 or pnl.std() == 0:
        return dict(sharpe=np.nan, sortino=np.nan, max_dd=np.nan)
    sharpe = pnl.mean() / pnl.std() * np.sqrt(periods_per_year)
    downside = pnl[pnl < 0].std()
    sortino = pnl.mean() / downside * np.sqrt(periods_per_year) if downside > 0 else np.nan
    eq = df['total_pnl']
    max_dd = float((eq.cummax() - eq).max())
    return dict(sharpe=float(sharpe), sortino=float(sortino), max_dd=max_dd)

rows = []
for label, df in ts.items():
    m = df.set_index('ts')[['total_pnl']].resample('1min').last().ffill()
    rows.append({'strategy': label, **annualized_metrics(m)})
pd.DataFrame(rows).set_index('strategy')

## 5. Что дальше

* Прогнать на полном `MD/lob.csv` + `MD/trades.csv` (не на sample).
* Подставить откалиброванные `A`, `k` в `configs/as*.ini` и сравнить.
* Параметрически свипнуть `gamma`, `order_size`, `max_inventory`,
  построить heatmap PnL/Sharpe.
* Добавить online-калибровку $\lambda(\delta)$ прямо в `BacktestEngine`
  (см. roadmap в `docs/DESIGN.md`).